# Australian Greyhound Racing — Exploratory Data Analysis

Ingests 4,179 daily Betfair BSP CSV files from Google Drive and produces:
1. **Data quality report** — coverage, completeness, venue/grade breakdown
2. **BSP calibration** — are the market odds well-calibrated?
3. **Signal exploration** — drift, volume, trap bias, grade effects
4. **Output** — cleaned master Parquet file for `greyhound_strategy.ipynb`

**Run in Google Colab with Drive mounted.**

In [ ]:
# Mount Google Drive (Colab only — skip if running locally)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Not in Colab — set DATA_DIR manually below')

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Paths ────────────────────────────────────────────────────────────────────
# Update DATA_DIR to the folder on your Drive containing the daily CSV files
if IN_COLAB:
    DATA_DIR   = '/content/drive/MyDrive/greyhound_data'   # <-- update if needed
else:
    DATA_DIR   = './greyhound_data'

OUTPUT_DIR = './greyhound_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Data directory : {DATA_DIR}')
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')))
print(f'CSV files found: {len(csv_files)}')
if csv_files:
    print(f'First: {os.path.basename(csv_files[0])}')
    print(f'Last : {os.path.basename(csv_files[-1])}')

## Format Diagnostics — inspect a 2024 file to identify column changes

In [ ]:
# Inspect first 2024 file to understand the new column format
# Run this BEFORE the bulk ingestion to detect any format changes

def inspect_file(path):
    name = os.path.basename(path)
    print(f'=== {name} ===')
    # Show raw first 3 lines
    with open(path, 'r', encoding='utf-8-sig', errors='replace') as f:
        for i, line in enumerate(f):
            print(f'  raw[{i}]: {line[:120].rstrip()}')
            if i >= 2:
                break
    # Try standard CSV parse and show columns
    for enc in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            df = pd.read_csv(path, dtype=str, nrows=3, encoding=enc,
                             on_bad_lines='skip')
            print(f'  encoding={enc} → {len(df.columns)} cols: {list(df.columns)}')
            break
        except Exception as e:
            print(f'  encoding={enc} → ERROR: {e}')
    print()

# Find one old-format and one new-format file
old_files = [p for p in csv_files if '2022' in os.path.basename(p) or '2021' in os.path.basename(p)]
new_files = [p for p in csv_files if '2024' in os.path.basename(p) or '2023' in os.path.basename(p)]

if old_files:
    inspect_file(old_files[0])
if new_files:
    inspect_file(new_files[0])
else:
    print('No 2023/2024 files found in file list')


DTYPES = {
    'EVENT_ID':          'int64',
    'WIN_LOSE':          'int8',
    'MORNINGTRADEDVOL':  'float32',
    'PPTRADEDVOL':       'float32',
    'IPTRADEDVOL':       'float32',
}

# Required columns in the canonical (old) format
REQUIRED_COLS = {'MENU_HINT', 'EVENT_NAME', 'WIN_LOSE', 'BSP', 'SELECTION_NAME'}

# Known Betfair format-v2 column renames (post ~2022).
# Keys = new name, Values = old canonical name we expect downstream.
# Extend this map if the diagnostic cell reveals other renamed columns.
COL_REMAP_V2 = {
    # Examples of columns Betfair has renamed in newer exports:
    'MARKET_TIME':        'EVENT_DT',
    'MARKET_ID':          'EVENT_ID',
    'SELECTION_BSP':      'BSP',
    'RUNNER_NAME':        'SELECTION_NAME',
    'RUNNER_ID':          'SELECTION_ID',
    'WIN_RESULT':         'WIN_LOSE',
    'LAST_TRADED_PRICE':  'BSP',          # some exports use LTP instead of BSP
    'CLASSIFICATION':     'MENU_HINT',
    'RACE_NAME':          'EVENT_NAME',
}

def parse_price(val):
    """Return float or NaN for missing/placeholder (empty string or '1' sentinel)."""
    try:
        f = float(val)
        return f if f > 1.01 else np.nan
    except (ValueError, TypeError):
        return np.nan

def _try_read(path):
    """Try reading with BOM-aware encoding; return DataFrame or None."""
    for enc in ('utf-8-sig', 'utf-8', 'latin-1'):
        try:
            df = pd.read_csv(path, dtype=str, low_memory=False,
                             on_bad_lines='skip', encoding=enc)
            return df
        except Exception:
            continue
    return None

def parse_file(path):
    df = _try_read(path)
    if df is None:
        print(f'  SKIP {os.path.basename(path)}: could not read file')
        return None

    # Normalise any v2 column names → canonical names
    df.rename(columns={k: v for k, v in COL_REMAP_V2.items() if k in df.columns},
              inplace=True)

    # Guard: skip if still missing required columns after remap
    if not REQUIRED_COLS.issubset(df.columns):
        missing = REQUIRED_COLS - set(df.columns)
        # Show actual columns on first occurrence to help extend COL_REMAP_V2
        if not hasattr(parse_file, '_unknown_shown'):
            parse_file._unknown_shown = set()
        col_sig = frozenset(df.columns)
        if col_sig not in parse_file._unknown_shown:
            parse_file._unknown_shown.add(col_sig)
            print(f'  SKIP {os.path.basename(path)}: missing {missing}')
            print(f'         actual cols: {list(df.columns)}')
        return None

    # Keep only AUS win markets (exclude forecast / combination markets)
    df = df[df['MENU_HINT'].str.contains('AUS', na=False)].copy()
    df = df[~df['EVENT_NAME'].str.match(r'^\d+ - ', na=False)]
    if df.empty:
        return None

    # Parse prices
    df['BSP']        = df['BSP'].apply(parse_price).astype('float32')
    df['MORNINGWAP'] = df['MORNINGWAP'].apply(parse_price).astype('float32') if 'MORNINGWAP' in df.columns else np.nan
    df['PPWAP']      = df['PPWAP'].apply(parse_price).astype('float32')      if 'PPWAP'      in df.columns else np.nan
    df['IPMAX']      = df['IPMAX'].apply(parse_price).astype('float32')      if 'IPMAX'      in df.columns else np.nan
    df['IPMIN']      = df['IPMIN'].apply(parse_price).astype('float32')      if 'IPMIN'      in df.columns else np.nan

    for col, dtype in DTYPES.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype)

    df['EVENT_DT'] = pd.to_datetime(df['EVENT_DT'], dayfirst=True, errors='coerce') if 'EVENT_DT' in df.columns else pd.NaT

    # Extract structured fields
    df['venue']    = df['MENU_HINT'].str.extract(r'/ (\w+) \(AUS\)')
    df['distance'] = df['EVENT_NAME'].str.extract(r'(\d{3,4})m').astype('float32')
    df['grade']    = df['EVENT_NAME'].str.extract(
        r'\b(Juv|Mdn|Gr[0-9]|FFA|Tr|Nov|Hdcp|Res)\b', expand=False
    )
    df['trap']     = df['SELECTION_NAME'].str.extract(r'^(\d+)\.').astype('float32')

    return df

print(f'Parsing {len(csv_files)} files...')
chunks  = []
skipped = 0
for i, path in enumerate(csv_files):
    chunk = parse_file(path)
    if chunk is not None:
        chunks.append(chunk)
    else:
        skipped += 1
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(csv_files)} files processed, {len(chunks)} with AUS data')

raw = pd.concat(chunks, ignore_index=True)
print(f'\nIngestion complete.')
print(f'  Total rows      : {len(raw):,}')
print(f'  Files skipped   : {skipped}')
print(f'  Date range      : {raw["EVENT_DT"].min().date()} → {raw["EVENT_DT"].max().date()}')
print(f'  Unique races    : {raw["EVENT_ID"].nunique():,}')
print(f'  Unique venues   : {raw["venue"].nunique()}')


In [ ]:
DTYPES = {
    'EVENT_ID':          'int64',
    'WIN_LOSE':          'int8',
    'MORNINGTRADEDVOL':  'float32',
    'PPTRADEDVOL':       'float32',
    'IPTRADEDVOL':       'float32',
}

# Expected columns — used to detect wrong-format files
REQUIRED_COLS = {'MENU_HINT', 'EVENT_NAME', 'WIN_LOSE', 'BSP', 'SELECTION_NAME'}

def parse_price(val):
    """Return float or NaN for missing/placeholder (empty string or '1' sentinel)."""
    try:
        f = float(val)
        return f if f > 1.01 else np.nan
    except (ValueError, TypeError):
        return np.nan

def parse_file(path):
    try:
        # on_bad_lines='skip' silently drops malformed rows (e.g. 18 fields instead of 17)
        df = pd.read_csv(path, dtype=str, low_memory=False, on_bad_lines='skip')
    except Exception as e:
        print(f'  SKIP {os.path.basename(path)}: {e}')
        return None

    # Guard: skip files missing required columns (different format / header)
    if not REQUIRED_COLS.issubset(df.columns):
        missing = REQUIRED_COLS - set(df.columns)
        print(f'  SKIP {os.path.basename(path)}: missing columns {missing}')
        return None

    # Keep only AUS win markets (exclude forecast / combination markets)
    df = df[df['MENU_HINT'].str.contains('AUS', na=False)].copy()
    df = df[~df['EVENT_NAME'].str.match(r'^\d+ - ', na=False)]   # drop forecast rows
    if df.empty:
        return None

    # Parse prices
    df['BSP']        = df['BSP'].apply(parse_price).astype('float32')
    df['MORNINGWAP'] = df['MORNINGWAP'].apply(parse_price).astype('float32')
    df['PPWAP']      = df['PPWAP'].apply(parse_price).astype('float32')
    df['IPMAX']      = df['IPMAX'].apply(parse_price).astype('float32')
    df['IPMIN']      = df['IPMIN'].apply(parse_price).astype('float32')

    for col, dtype in DTYPES.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(dtype)

    df['EVENT_DT'] = pd.to_datetime(df['EVENT_DT'], dayfirst=True, errors='coerce')

    # Extract structured fields
    df['venue']    = df['MENU_HINT'].str.extract(r'/ (\w+) \(AUS\)')
    df['distance'] = df['EVENT_NAME'].str.extract(r'(\d{3,4})m').astype('float32')
    df['grade']    = df['EVENT_NAME'].str.extract(
        r'\b(Juv|Mdn|Gr[0-9]|FFA|Tr|Nov|Hdcp|Res)\b', expand=False
    )
    df['trap']     = df['SELECTION_NAME'].str.extract(r'^(\d+)\.').astype('float32')

    return df

print(f'Parsing {len(csv_files)} files...')
chunks = []
skipped = 0
for i, path in enumerate(csv_files):
    chunk = parse_file(path)
    if chunk is not None:
        chunks.append(chunk)
    else:
        skipped += 1
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(csv_files)} files processed, {len(chunks)} with AUS data')

raw = pd.concat(chunks, ignore_index=True)
print(f'\nIngestion complete.')
print(f'  Total rows      : {len(raw):,}')
print(f'  Files skipped   : {skipped}')
print(f'  Date range      : {raw["EVENT_DT"].min().date()} → {raw["EVENT_DT"].max().date()}')
print(f'  Unique races    : {raw["EVENT_ID"].nunique():,}')
print(f'  Unique venues   : {raw["venue"].nunique()}')


## Phase 2 — Data Quality

In [ ]:
total = len(raw)
print('Column completeness:')
for col in ['BSP', 'MORNINGWAP', 'PPWAP', 'MORNINGTRADEDVOL', 'venue', 'distance', 'grade', 'trap']:
    n = raw[col].notna().sum()
    print(f'  {col:<20s}: {n:>8,}  ({n/total*100:.1f}%)')

print(f'\nRunner count per race:')
runners_per_race = raw.groupby('EVENT_ID').size()
print(runners_per_race.value_counts().sort_index().to_string())

print(f'\nTop 20 venues by race count:')
venue_counts = raw.drop_duplicates('EVENT_ID').groupby('venue').size().sort_values(ascending=False)
print(venue_counts.head(20).to_string())

print(f'\nGrade distribution:')
print(raw.drop_duplicates('EVENT_ID')['grade'].value_counts(dropna=False).head(15).to_string())

print(f'\nDistance distribution:')
print(raw.drop_duplicates('EVENT_ID')['distance'].value_counts(dropna=False).sort_index().to_string())

In [ ]:
# Timeline: races per month
raw['year_month'] = raw['EVENT_DT'].dt.to_period('M')
monthly = raw.drop_duplicates('EVENT_ID').groupby('year_month').size()

fig, ax = plt.subplots(figsize=(14, 3))
monthly.plot(ax=ax, color='steelblue', linewidth=1.2)
ax.set_title('AUS Greyhound Races per Month')
ax.set_xlabel('')
ax.set_ylabel('Races')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## Phase 3 — Build Race-Level Table

Reshape from one row per runner to one row per race with runners ranked by BSP (favourite first).

In [ ]:
def build_race_table(df):
    """
    For each race, rank runners by BSP (then MORNINGWAP if BSP missing).
    Returns a race-level DataFrame with favourite info and winner info.
    """
    records = []
    for eid, group in df.groupby('EVENT_ID'):
        n = len(group)
        winner = group[group['WIN_LOSE'] == 1]
        if winner.empty or n < 4:
            continue

        # Use BSP; fall back to MORNINGWAP for ranking
        group = group.copy()
        group['rank_price'] = group['BSP'].fillna(group['MORNINGWAP'])
        if group['rank_price'].isna().all():
            continue

        group = group.sort_values('rank_price')   # lowest price = favourite
        group['mkt_rank'] = range(1, len(group) + 1)

        fav   = group.iloc[0]
        win_r = winner.iloc[0]

        rec = {
            'event_id':          eid,
            'event_dt':          group['EVENT_DT'].iloc[0],
            'venue':             group['venue'].iloc[0],
            'distance':          group['distance'].iloc[0],
            'grade':             group['grade'].iloc[0],
            'n_runners':         n,
            # Favourite
            'fav_bsp':           fav['BSP'],
            'fav_morningwap':    fav['MORNINGWAP'],
            'fav_trap':          fav['trap'],
            'fav_morning_vol':   fav['MORNINGTRADEDVOL'],
            'fav_won':           int(fav['WIN_LOSE']),
            # Winner
            'winner_bsp':        win_r['BSP'],
            'winner_morningwap': win_r['MORNINGWAP'],
            'winner_trap':       win_r['trap'],
            'winner_mkt_rank':   int(group.loc[win_r.name, 'mkt_rank']),
            # Market-level
            'total_morning_vol': group['MORNINGTRADEDVOL'].sum(),
            'has_bsp':           int(group['BSP'].notna().any()),
        }

        # Per-runner BSP and drift for up to 8 runners
        for i, (_, runner) in enumerate(group.iterrows(), 1):
            rec[f'r{i}_bsp']      = runner['BSP']
            rec[f'r{i}_mwap']     = runner['MORNINGWAP']
            rec[f'r{i}_trap']     = runner['trap']
            rec[f'r{i}_won']      = int(runner['WIN_LOSE'])
            rec[f'r{i}_vol']      = runner['MORNINGTRADEDVOL']
            if i >= 8:
                break

        records.append(rec)

    return pd.DataFrame(records)

print('Building race-level table...')
races = build_race_table(raw)

# Derived signals
races['fav_drift']        = (races['fav_morningwap'] - races['fav_bsp']) / races['fav_morningwap']
races['fav_implied_prob'] = 1.0 / races['fav_bsp']
races['fav_morning_prob'] = 1.0 / races['fav_morningwap']
races['year']             = races['event_dt'].dt.year
races['month']            = races['event_dt'].dt.month
races['day_of_week']      = races['event_dt'].dt.dayofweek

print(f'Race-level table: {len(races):,} races')
print(f'  With BSP      : {races["has_bsp"].sum():,} ({races["has_bsp"].mean()*100:.1f}%)')
print(f'  Favourite win : {races["fav_won"].mean()*100:.1f}%')
print(races.head(3).to_string())

## Phase 4 — BSP Calibration

Are the Betfair Starting Prices well-calibrated for Australian greyhounds?
If actual win rate > implied prob at some odds range, there's a backing edge. If lower, a laying edge.

In [ ]:
# Use all runners (not just favourites) for calibration
bsp_rows = raw[(raw['BSP'].notna()) & (raw['BSP'] > 1.01)].copy()
bsp_rows['implied_prob'] = 1.0 / bsp_rows['BSP']

# Bucket by odds
bins  = [1, 1.5, 2, 2.5, 3, 4, 5, 7, 10, 15, 20, 30, 100]
bsp_rows['odds_bucket'] = pd.cut(bsp_rows['BSP'], bins=bins)

cal = bsp_rows.groupby('odds_bucket', observed=True).agg(
    n         = ('WIN_LOSE', 'count'),
    actual_wr = ('WIN_LOSE', 'mean'),
    mid_bsp   = ('BSP', 'median'),
).reset_index()
cal['implied_wr'] = 1.0 / cal['mid_bsp']
cal['edge']       = cal['actual_wr'] - cal['implied_wr']
cal['edge_pct']   = cal['edge'] * 100

print('BSP Calibration by Odds Bucket:')
print(f'{"Odds range":>20s} {"N":>7s} {"Implied%":>9s} {"Actual%":>9s} {"Edge pp":>8s}')
print('-' * 57)
for _, r in cal.iterrows():
    flag = ' ◄' if abs(r['edge_pct']) > 1.5 else ''
    print(f'{str(r["odds_bucket"]):>20s} {int(r["n"]):>7,} '
          f'{r["implied_wr"]*100:>8.1f}% {r["actual_wr"]*100:>8.1f}% '
          f'{r["edge_pct"]:>+7.2f}pp{flag}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('BSP Calibration — Australian Greyhounds', fontsize=13)

# Left: calibration scatter
ax = axes[0]
ax.scatter(cal['implied_wr']*100, cal['actual_wr']*100,
           s=cal['n']/cal['n'].max()*400 + 30, color='steelblue', alpha=0.8,
           zorder=5, label='Odds bucket (size = N)')
diag = [0, 70]
ax.plot(diag, diag, 'k--', linewidth=1, label='Perfect calibration')
ax.set_xlabel('BSP implied win % (1/odds)')
ax.set_ylabel('Actual win %')
ax.set_title('Actual vs BSP Implied Win Rate')
ax.legend(fontsize=8)

# Right: edge by odds bucket
ax2 = axes[1]
colors_edge = ['#2ecc71' if e > 0 else '#e74c3c' for e in cal['edge_pct']]
ax2.bar(range(len(cal)), cal['edge_pct'], color=colors_edge, alpha=0.85)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_xticks(range(len(cal)))
ax2.set_xticklabels([str(b) for b in cal['odds_bucket']], rotation=45, ha='right', fontsize=7)
ax2.set_title('Edge (Actual − Implied) by Odds Bucket')
ax2.set_ylabel('Edge (percentage points)')
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.1f%%'))

plt.tight_layout()
plt.show()

## Phase 5 — Favourite Analysis

In [ ]:
bsp_races = races[races['fav_bsp'].notna()].copy()

# Overall favourite stats
print('=== Favourite Overview ===')
print(f'Races with BSP favourite     : {len(bsp_races):,}')
print(f'Favourite win rate           : {bsp_races["fav_won"].mean()*100:.1f}%')
print(f'Favourite BSP median         : ${bsp_races["fav_bsp"].median():.2f}')
print(f'Favourite BSP mean           : ${bsp_races["fav_bsp"].mean():.2f}')

# Win rate by n_runners
print('\nFavourite win rate by field size:')
print(bsp_races.groupby('n_runners')['fav_won'].agg(['mean','count'])
      .rename(columns={'mean':'win_rate','count':'n'})
      .assign(win_rate=lambda d: (d['win_rate']*100).round(1))
      .to_string())

# Win rate by grade
print('\nFavourite win rate by grade:')
print(bsp_races.groupby('grade')['fav_won'].agg(['mean','count'])
      .rename(columns={'mean':'win_rate','count':'n'})
      .assign(win_rate=lambda d: (d['win_rate']*100).round(1))
      .sort_values('win_rate', ascending=False)
      .to_string())

# Win rate by distance bucket
bsp_races['dist_bucket'] = pd.cut(bsp_races['distance'],
                                   bins=[0,400,480,560,640,800],
                                   labels=['<400m','400-480m','480-560m','560-640m','>640m'])
print('\nFavourite win rate by distance:')
print(bsp_races.groupby('dist_bucket', observed=True)['fav_won'].agg(['mean','count'])
      .rename(columns={'mean':'win_rate','count':'n'})
      .assign(win_rate=lambda d: (d['win_rate']*100).round(1))
      .to_string())

## Phase 6 — Trap Bias

In [ ]:
# Overall trap win rate
trap_rows = raw[raw['trap'].notna() & (raw['trap'] <= 8)].copy()
trap_win = trap_rows.groupby('trap')['WIN_LOSE'].agg(['mean','count']).reset_index()
trap_win.columns = ['trap','win_rate','n']
expected = 1 / 8

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Trap Bias — Australian Greyhounds', fontsize=13)

ax = axes[0]
colors_trap = ['#2ecc71' if w > expected else '#e74c3c' for w in trap_win['win_rate']]
ax.bar(trap_win['trap'], trap_win['win_rate']*100, color=colors_trap, alpha=0.85)
ax.axhline(expected*100, color='black', linewidth=1, linestyle='--', label=f'Equal ({expected*100:.1f}%)')
ax.set_xlabel('Trap')
ax.set_ylabel('Win rate (%)')
ax.set_title('Win Rate by Trap (all races)')
ax.legend()
ax.set_xticks(range(1,9))

# Trap win rate by distance bucket
ax2 = axes[1]
dist_trap = raw[raw['trap'].notna() & raw['distance'].notna()].copy()
dist_trap['dist_bucket'] = pd.cut(dist_trap['distance'],
                                    bins=[0,480,560,800],
                                    labels=['Short <480m','Mid 480-560m','Long >560m'])
pivot = dist_trap.groupby(['dist_bucket','trap'], observed=True)['WIN_LOSE'].mean().unstack('trap') * 100
pivot.T.plot(ax=ax2, marker='o', linewidth=1.5)
ax2.axhline(expected*100, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Trap')
ax2.set_ylabel('Win rate (%)')
ax2.set_title('Trap Win Rate by Distance')
ax2.legend(fontsize=8)
ax2.set_xticks(range(1,9))

plt.tight_layout()
plt.show()

print('Trap win rate (all races):')
for _, r in trap_win.iterrows():
    bar = '█' * int(r['win_rate']*400)
    print(f'  Trap {int(r["trap"])}: {r["win_rate"]*100:5.2f}%  {bar}')

## Phase 7 — Morning Price Drift Signal

In [ ]:
drift_races = bsp_races[bsp_races['fav_morningwap'].notna() &
                         bsp_races['fav_bsp'].notna()].copy()

# drift > 0 = price shortened (shortener), drift < 0 = drifted out
drift_races['fav_drift'] = (drift_races['fav_morningwap'] - drift_races['fav_bsp']) / drift_races['fav_morningwap']

# Bucket drift
drift_races['drift_bucket'] = pd.cut(
    drift_races['fav_drift'],
    bins=[-2, -0.30, -0.15, -0.05, 0.05, 0.15, 0.30, 2],
    labels=['Strong drift', 'Drift', 'Slight drift', 'Stable',
            'Slight shorten', 'Shorten', 'Strong shorten']
)

drift_stats = drift_races.groupby('drift_bucket', observed=True)['fav_won'].agg(
    win_rate='mean', n='count'
).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
colors_drift = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(drift_stats)))
bars = ax.bar(drift_stats['drift_bucket'], drift_stats['win_rate']*100,
               color=colors_drift, alpha=0.85)
ax.axhline(drift_races['fav_won'].mean()*100, color='black', linewidth=1,
            linestyle='--', label='Overall avg')
for bar, n in zip(bars, drift_stats['n']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'n={n:,}', ha='center', va='bottom', fontsize=7)
ax.set_xlabel('Favourite price movement (Morning → BSP)')
ax.set_ylabel('Favourite win rate (%)')
ax.set_title('Does Morning Price Drift Predict Winner?')
ax.legend()
plt.tight_layout()
plt.show()

print(drift_stats[['drift_bucket','n','win_rate']]
      .assign(win_rate=lambda d: (d['win_rate']*100).round(2))
      .to_string(index=False))

## Phase 8 — Market Volume Signal

In [ ]:
vol_races = bsp_races[bsp_races['total_morning_vol'] > 0].copy()
vol_races['vol_bucket'] = pd.qcut(vol_races['total_morning_vol'], q=5,
                                    labels=['Very low','Low','Medium','High','Very high'])

vol_stats = vol_races.groupby('vol_bucket', observed=True).agg(
    fav_win_rate  = ('fav_won', 'mean'),
    n             = ('fav_won', 'count'),
    median_vol    = ('total_morning_vol', 'median'),
    median_fav_bsp = ('fav_bsp', 'median'),
).reset_index()

print('Favourite performance by morning market volume:')
print(vol_stats.assign(fav_win_rate=lambda d: (d['fav_win_rate']*100).round(1)).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(vol_stats['vol_bucket'], vol_stats['fav_win_rate']*100,
       color='steelblue', alpha=0.85)
ax.axhline(bsp_races['fav_won'].mean()*100, color='black', linewidth=1,
            linestyle='--', label='Overall avg')
ax.set_xlabel('Morning market volume (quintile)')
ax.set_ylabel('Favourite win rate (%)')
ax.set_title('Favourite Win Rate by Market Volume')
ax.legend()
plt.tight_layout()
plt.show()

## Phase 9 — Venue-Level Analysis

In [ ]:
venue_stats = bsp_races.groupby('venue').agg(
    n_races        = ('event_id', 'count'),
    fav_win_rate   = ('fav_won', 'mean'),
    median_fav_bsp = ('fav_bsp', 'median'),
).query('n_races >= 50').sort_values('fav_win_rate', ascending=False)

fig, ax = plt.subplots(figsize=(14, 5))
colors_v = ['#2ecc71' if w > bsp_races['fav_won'].mean() else '#e74c3c'
             for w in venue_stats['fav_win_rate']]
ax.bar(venue_stats.index, venue_stats['fav_win_rate']*100, color=colors_v, alpha=0.85)
ax.axhline(bsp_races['fav_won'].mean()*100, color='black', linewidth=1,
            linestyle='--', label=f'Overall ({bsp_races["fav_won"].mean()*100:.1f}%)')
ax.set_xlabel('Venue')
ax.set_ylabel('Favourite win rate (%)')
ax.set_title('Favourite Win Rate by Venue (min 50 races)')
ax.legend()
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

print(venue_stats.assign(fav_win_rate=lambda d: (d['fav_win_rate']*100).round(1)).to_string())

## Phase 10 — Winner Rank Distribution

In [ ]:
rank_dist = bsp_races['winner_mkt_rank'].value_counts().sort_index()
rank_pct  = rank_dist / rank_dist.sum() * 100

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(rank_pct.index, rank_pct.values,
       color=plt.cm.viridis(np.linspace(0.8, 0.2, len(rank_pct))), alpha=0.85)
ax.set_xlabel('Market rank of winner (1 = BSP favourite)')
ax.set_ylabel('% of races won')
ax.set_title('Winner Market Rank Distribution')
ax.set_xticks(rank_pct.index)
for i, (rank, pct) in enumerate(rank_pct.items()):
    ax.text(rank, pct + 0.3, f'{pct:.1f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

cumulative = rank_pct.cumsum()
print('Cumulative win % by market rank:')
for rank, cum in cumulative.items():
    print(f'  Top {rank}: {cum:.1f}%')

## Save Outputs

In [ ]:
# Save master runner-level table and race-level table
runner_path = os.path.join(OUTPUT_DIR, 'greyhound_runners.parquet')
race_path   = os.path.join(OUTPUT_DIR, 'greyhound_races.parquet')
cal_path    = os.path.join(OUTPUT_DIR, 'bsp_calibration.csv')

raw.to_parquet(runner_path, index=False)
races.to_parquet(race_path, index=False)
cal.to_csv(cal_path, index=False)

print(f'Saved:')
print(f'  {runner_path}  ({len(raw):,} rows)')
print(f'  {race_path}    ({len(races):,} rows)')
print(f'  {cal_path}')
print(f'\nPass greyhound_races.parquet to greyhound_strategy.ipynb')